# EDA — Ultimate Offensive Red Team

**Dataset:** [`WNT3D/Ultimate-Offensive-Red-Team`](https://huggingface.co/datasets/WNT3D/Ultimate-Offensive-Red-Team)

Colección de datos de seguridad ofensiva (herramientas, técnicas, CVE, smart contracts) en formato instrucción/entrada/salida, más varios JSON de referencia.

**Formato:** Muchos archivos: varios `.jsonl` (`instruction`, `input`, `output`), un `.parquet` y JSON anidados.

**Preguntas guía para el EDA**
- ¿Qué contiene cada archivo y cuánto pesa?
- ¿Los distintos `.jsonl` se repiten entre sí?
- ¿Qué tan 'plantillado' es el contenido? (instrucciones idénticas, input == output)
- ¿Qué estructura tienen los JSON de referencia?

In [ ]:
import sys, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Busca la raíz del repo (donde está src/profiler.py) y la agrega al path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "profiler.py").exists())
sys.path.insert(0, str(ROOT))
from src.profiler import *

sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

ctx = eda_context("07")   # rutas: datos en data/raw/..., resultados en esta carpeta
print(ctx)

## 1. Inventario de archivos

In [ ]:
files = list_files(ctx.data_dir)
display(summarize_extensions(files))
display(files)

## 2. Resumen de cada archivo tabular

In [ ]:
rows = []
for p in sorted(ctx.data_dir.glob("*")):
    if p.suffix not in (".jsonl", ".parquet"):
        continue
    n = count_lines(p) if p.suffix == ".jsonl" else len(pd.read_parquet(p))
    cols = list(peek_record(p).keys()) if p.suffix == ".jsonl" else list(pd.read_parquet(p).columns)
    rows.append({"archivo": p.name, "size_mb": round(p.stat().st_size / 1e6, 2), "filas": n, "columnas": cols})
tab = pd.DataFrame(rows).sort_values("size_mb", ascending=False)
display(tab)

## 3. Cargar los `.jsonl` (el más grande, como muestra)

In [ ]:
SAMPLE_N = 50_000
frames = []
for p in sorted(ctx.data_dir.glob("*.jsonl")):
    d = read_jsonl(p, sample=SAMPLE_N) if p.stat().st_size > 50e6 else read_jsonl(p)
    d["archivo"] = p.name
    frames.append(d)
pq = pd.read_parquet(ctx.data_dir / "training_data.parquet"); pq["archivo"] = "training_data.parquet"
frames.append(pq)
df = pd.concat(frames, ignore_index=True)
display(df.groupby("archivo").size().to_frame("filas_cargadas"))
display(profile_df(df))

## 4. Señales de datos plantillados o repetidos

In [ ]:
for c in ["instruction", "input", "output"]:
    df[c] = df[c].astype(str)
g = df.groupby("archivo").agg(
    filas=("instruction", "size"),
    instr_unicas=("instruction", "nunique"),
    input_unicos=("input", "nunique"),
    output_unicos=("output", "nunique"),
    input_igual_output=("input", lambda s: (s == df.loc[s.index, "output"]).mean()),
)
g["%output_unico"] = (100 * g.output_unicos / g.filas).round(1)
display(g)

display(df["instruction"].str.replace(r"\d+", "#", regex=True).value_counts().head(20).to_frame("n"))
display(df["output"].value_counts().head(10).to_frame("n"))

In [ ]:
# ¿Se repiten registros entre archivos?
df["h"] = (df["instruction"] + "||" + df["input"] + "||" + df["output"]).map(text_hash)
por_archivo = df.groupby("archivo")["h"].apply(set)
names = list(por_archivo.index)
M = pd.DataFrame([[len(por_archivo[a] & por_archivo[b]) / max(len(por_archivo[a]), 1) for b in names] for a in names],
                 index=names, columns=names)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(M, annot=True, fmt=".0%", cmap="Blues", ax=ax)
ax.set_title("% de registros de la fila que también están en la columna")
save_fig(fig, ctx, "solapamiento_archivos"); plt.show()
df = df.drop(columns=["h"])

## 5. JSON de referencia (estructura)

In [ ]:
for p in sorted(ctx.data_dir.glob("*.json")):
    data = read_json_any(p)
    print(f"\n=== {p.name} ({p.stat().st_size/1e6:.2f} MB) ===")
    display(describe_json_structure(data, max_depth=2).head(25))

## 6. Ejemplos

In [ ]:
show_examples(df, n=3, cols=["archivo", "instruction", "input", "output"], max_chars=500)

## Guardar resultados
Se guardan en `tables/` de esta carpeta. `resumen.csv` lo consolida `eda/00_general`.

In [ ]:
save_table(profile_df(df), ctx, "perfil_columnas")
display(save_summary(ctx, "WNT3D/Ultimate-Offensive-Red-Team", df, total_rows=None, dup_cols=["instruction", "input", "output"],
                     notas="jsonl grandes muestreados"))

## Próximos pasos / notas
- Anota tus hallazgos en el `README.md` de esta carpeta.
-